# 02 - Preprocessing
Create a clean regular time series and save it under `data/processed/`.

In [107]:
import sys
from pathlib import Path
import pandas as pd
sys.path.append('..')

df = pd.read_csv('../data/raw/retail_store_inventory.csv')


#from src.preprocessing import load_data, prepare_time_series, save_processed_data

#df = load_data('../data/raw/retail_store_inventory.csv')
#ts = prepare_time_series(df, date_col='Date', target_col='Sales')
#save_processed_data(ts, '../data/processed/daily_sales.csv')
#ts.head()

In [108]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.width', 500)

In [109]:
# train test split yapılmadan önce yapulacaklar:
    # Tarihten Year, Month, Day, DayOfWeek veya Seasonality_NEW türetme.
    # Hatalı gürültü değişkenini (Seasonality) düşürme.
    # Satır bazlı matematiksel işlemler: Net_Price = Price * (1 - Discount/100) veya Price_Diff.
    # Demand Forecast sütunundaki negatif değerleri (-9.99) minimum 0 yapma.

In [110]:
# Date kolonunu objectten datetime64[ns] tipine çevirme
df['Date'] = df['Date'].astype('datetime64[ns]')

In [111]:
# Zaman Serisi Kronolojik Sıralama
# Sıralama bozuk olursa, shift(1) işlemi yanlış güne denk gelebilir
df = df.sort_values(by=['Store ID', 'Product ID', 'Date']).reset_index(drop=True)

In [112]:
# Seasonality kolonu artık gereksiz ve gürültü değişkeni olduğu için düşürülüyor.
df.drop(["Seasonality"], axis=1, inplace=True)

In [113]:
# Demand Forecast'i dropla data leakage'a sebep oluyor
df = df.drop(columns=['Demand Forecast'])

In [114]:
# Year - Month - Day- DayOfWeek feature'larını türetme
import datetime as dt
def date_features(dataframe):
    dataframe['Year'] = dataframe['Date'].dt.year
    dataframe['Month'] = dataframe['Date'].dt.month
    dataframe['Day'] = dataframe['Date'].dt.day
    dataframe['DayOfWeek'] = dataframe['Date'].dt.dayofweek
    return dataframe

df = date_features(df)

In [115]:
#Seanolity sentetik bir şekilde rastgele üretilmişti. 
# Bu nedenle, gerçek aya göre türetmek daha mantıklı olabilir.

df["Seasonality_NEW"] = df["Date"].dt.month.map({
                                                12: "Winter", 1: "Winter", 2: "Winter",
                                                3: "Spring", 4: "Spring", 5: "Spring",
                                                6: "Summer", 7: "Summer", 8: "Summer",
                                                9: "Fall", 10: "Fall", 11: "Fall"})

In [116]:
# Price'a Discount uygulanarak Net_Price hesaplanması
df["NET_PRICE"] = df["Price"] * (1 - df["Discount"]/100)

In [117]:
# Competitor Pricing sütunu, ürünün aynı gün rakipler tarafından hangi fiyattan satıldığını gösteren değişken. 
# Correlation mapte Price ile correlation'ı çok yüksek sorun çıkaracağı için yeni column türetip salcaz. 
# (multicollinearity sorunu)

df["Price_Diff"] = df["Price"] - df["Competitor Pricing"]
df["Price_Ratio"] = df["Price"] / df["Competitor Pricing"]

In [118]:
# Yukarıdaki 2 column eklendikten sonra multicol problemi yaşanmasın diye Competitor Pricing'i siliyoruz
df.drop(["Competitor Pricing"], axis=1, inplace=True)

In [119]:
# Lag feature'ları türet
# Store ve Product ID kırılımına göre üret böylece o mağazada o ürünün geçmiş s
# atışları ile tahmin yapılabilir.

def lag_features(dataframe, lags):
    for lag in lags:
        dataframe['units_sold_lag_' + str(lag)] = dataframe.groupby(["Store ID", "Product ID"])['Units Sold'].transform(
            lambda x: x.shift(lag)) 
    return dataframe

df = lag_features(df, [1, 2, 3, 7, 14, 30, 365])

In [120]:
# rolling mean feature'ları türet
def roll_mean_features(dataframe, windows):
    for window in windows:
        dataframe['sales_roll_mean_' + str(window)] = (
            dataframe.groupby(["Store ID", "Product ID"])['Units Sold']
            .transform(lambda x: x.shift(1).rolling(window=window).mean())
        )
    return dataframe
df = roll_mean_features(df, [7, 30])

In [121]:
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Holiday/Promotion,Year,Month,Day,DayOfWeek,Seasonality_NEW,NET_PRICE,Price_Diff,Price_Ratio,units_sold_lag_1,units_sold_lag_2,units_sold_lag_3,units_sold_lag_7,units_sold_lag_14,units_sold_lag_30,units_sold_lag_365,sales_roll_mean_7,sales_roll_mean_30
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,33.500,20,Rainy,0,2022,1,1,5,Winter,26.800,3.810,1.128,nan,nan,nan,nan,nan,nan,nan,nan,nan
1,2022-01-02,S001,P0001,Groceries,West,116,81,104,27.950,10,Cloudy,0,2022,1,2,6,Winter,25.155,-2.940,0.905,127.000,nan,nan,nan,nan,nan,nan,nan,nan
2,2022-01-03,S001,P0001,Electronics,West,154,5,189,62.700,20,Rainy,0,2022,1,3,0,Winter,50.160,4.480,1.077,81.000,127.000,nan,nan,nan,nan,nan,nan,nan
3,2022-01-04,S001,P0001,Groceries,South,85,58,193,77.880,15,Cloudy,1,2022,1,4,1,Winter,66.198,1.890,1.025,5.000,81.000,127.000,nan,nan,nan,nan,nan,nan
4,2022-01-05,S001,P0001,Groceries,South,238,147,37,28.460,20,Sunny,1,2022,1,5,2,Winter,22.768,-0.940,0.968,58.000,5.000,81.000,nan,nan,nan,nan,nan,nan


In [122]:
df["Discount"].value_counts()

20    14715
0     14662
15    14624
5     14591
10    14508
Name: Discount, dtype: int64

In [123]:
cat_cols = ['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality_NEW', 'Holiday/Promotion']

num_cols = [ 'Discount','Inventory Level', 'Units Ordered', 'Price',  'Month', 'Day', 'NET_PRICE', 'units_sold_lag_1', 'units_sold_lag_2', 'units_sold_lag_3', 'units_sold_lag_7', 'units_sold_lag_14', 'units_sold_lag_30', 'units_sold_lag_365', 'sales_roll_mean_7', 'sales_roll_mean_30', 'Year', 'DayOfWeek', 'Price_Diff',  'Price_Ratio']

In [124]:
print(f"Categorical Columns: {cat_cols}")
print(f"Numerical Columns: {num_cols}")

Categorical Columns: ['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality_NEW', 'Holiday/Promotion']
Numerical Columns: ['Discount', 'Inventory Level', 'Units Ordered', 'Price', 'Month', 'Day', 'NET_PRICE', 'units_sold_lag_1', 'units_sold_lag_2', 'units_sold_lag_3', 'units_sold_lag_7', 'units_sold_lag_14', 'units_sold_lag_30', 'units_sold_lag_365', 'sales_roll_mean_7', 'sales_roll_mean_30', 'Year', 'DayOfWeek', 'Price_Diff', 'Price_Ratio']


In [ ]:
# Test ve train setlerini ayır 
    # Bu bir zaman serisi tahmin problemidir. Bu nedenle, verileri 
    # rastgele ayırmak yerine, belirli bir tarih noktasına göre ayırmak gereklidir

# Dataset 2022-01-01 - 2024-01-01 arasını kapsar. Son 30 gün test seti olarak ayırmak mantıklı olabilir. 
# Bu nedenle, 2023-12-01 tarihinden itibaren olan veriler test seti olarak ayrılabilir.

train = df[df['Date'] < '2023-11-01']
validation = df[(df["Date"] >= "2023-11-01") & (df["Date"] < "2023-12-01")]
test = df[df['Date'] >= '2023-12-01']


In [ ]:
train.shape, test.shape, validation.shape

((69900, 29), (3200, 29))

In [127]:
# One-Hot-Encoding ID'lere yapılmamalı, kardinalitisi çok yüksek
# Curse of Dimensionality sorunu yaratır, Bu yüzden OHE yapılacak kategorik değişkenleri
# seçtik

cat_cols_OHE = ['Category', 'Region', 'Weather Condition', 'Seasonality_NEW']

In [ ]:
# Train test olarak ayırıldıktan sonra ilk olarak cat collara one hot yapılacak


#  Train ve Test kümesinde get_dummies uygula
train_ohe_df = pd.get_dummies(train[cat_cols_OHE], drop_first=True)
test_ohe_df = pd.get_dummies(test[cat_cols_OHE], drop_first=True)
validation_ohe_df = pd.get_dummies(validation[cat_cols_OHE], drop_first=True)


#  Trainde olan her şey gelicek, test'te var olanlar gelicek olmayanlar drop olcak
# Yani referans olarak train seti alınacak. Böylecce train testi görmemiş olacak
train_ohe_df, validation_ohe_df = train_ohe_df.align(validation_ohe_df,join='left',axis=1,fill_value=0)

train_ohe_df, test_ohe_df = train_ohe_df.align(test_ohe_df,join='left',axis=1,fill_value=0)

# 3. Eski kategorik metin kolonlarını drop edip yenileriyle birleştir
final_train_df = pd.concat([train.drop(columns=cat_cols_OHE), train_ohe_df], axis=1)
final_test_df = pd.concat([test.drop(columns=cat_cols_OHE), test_ohe_df], axis=1)
final_validation_df = pd.concat([validation.drop(columns=cat_cols_OHE), validation_ohe_df],axis=1)

Dummy variable 1 sütun düşürür. OHE yapılırken 1 eksik column oluşur yani hepsi 0 0 0
ise yazılmayan sütundur. 
Region_East, Region_West, Region_North hepsi 0 ise demek ki Southtır. kolon sayısı artmasın diye South için ayrı kolon oluşmaz.

!!!!! Bunun için 40 saat düşündüm !!!!

In [ ]:

cat_features = ['Store ID', 'Product ID']
# ID'ler cardinality olarak çoktur OHE yapılamaz ama eğitim ve test için de gereklidir.
# Label Encoding 0-1-2 gibi ordinal değer atar ama ID'ler birbirinden üstün değildir
# LightGBM gibi modeller için dönüşüm şarttır bu yüzden bu veriler categoric olarak atanır
# arka planda Pandas her ID'ye  bir tamsayı kodu atar
for col in cat_features:

    final_train_df[col] = final_train_df[col].astype('category')

    categories = final_train_df[col].cat.categories

    final_validation_df[col] = pd.Categorical(final_validation_df[col], categories=categories)

    final_test_df[col] = pd.Categorical(final_test_df[col], categories=categories)

In [130]:
final_test_df.head()

,Date,Store ID,Product ID,Inventory Level,Units Sold,Units Ordered,Price,Discount,Holiday/Promotion,Year,Month,Day,DayOfWeek,NET_PRICE,Price_Diff,Price_Ratio,units_sold_lag_1,units_sold_lag_2,units_sold_lag_3,units_sold_lag_7,units_sold_lag_14,units_sold_lag_30,units_sold_lag_365,sales_roll_mean_7,sales_roll_mean_30,Category_Electronics,Category_Furniture,Category_Groceries,Category_Toys,Region_North,Region_South,Region_West,Weather Condition_Rainy,Weather Condition_Snowy,Weather Condition_Sunny,Seasonality_NEW_Spring,Seasonality_NEW_Summer,Seasonality_NEW_Winter
699,2023-12-01,S001,P0001,397,185,58,59.790,10,1,2023,12,1,4,53.811,-2.880,0.954,244.000,110.000,89.000,16.000,416.000,15.000,191.000,152.714,137.700,0,0,1,0,0,0,1,0,0,0,0,0,0
700,2023-12-02,S001,P0001,214,200,136,60.900,20,0,2023,12,2,5,48.720,0.460,1.008,185.000,244.000,110.000,371.000,223.000,139.000,289.000,176.857,143.367,0,0,0,1,0,0,0,0,0,0,0,0,0
701,2023-12-03,S001,P0001,482,122,81,21.920,0,0,2023,12,3,6,21.920,0.040,1.002,200.000,185.000,244.000,52.000,105.000,136.000,7.000,152.429,145.400,0,0,1,0,0,1,0,0,0,1,0,0,0
702,2023-12-04,S001,P0001,57,49,26,71.240,0,0,2023,12,4,0,71.240,-4.700,0.938,122.000,200.000,185.000,187.000,365.000,180.000,350.000,162.429,144.933,0,0,0,0,0,1,0,1,0,0,0,0,0
703,2023-12-05,S001,P0001,104,66,122,44.770,0,1,2023,12,5,1,44.770,-3.390,0.930,49.000,122.000,200.000,89.000,21.000,21.000,6.000,142.714,140.567,1,0,0,0,1,0,0,0,0,1,0,0,0


In [131]:
final_train_df.head()

,Date,Store ID,Product ID,Inventory Level,Units Sold,Units Ordered,Price,Discount,Holiday/Promotion,Year,Month,Day,DayOfWeek,NET_PRICE,Price_Diff,Price_Ratio,units_sold_lag_1,units_sold_lag_2,units_sold_lag_3,units_sold_lag_7,units_sold_lag_14,units_sold_lag_30,units_sold_lag_365,sales_roll_mean_7,sales_roll_mean_30,Category_Electronics,Category_Furniture,Category_Groceries,Category_Toys,Region_North,Region_South,Region_West,Weather Condition_Rainy,Weather Condition_Snowy,Weather Condition_Sunny,Seasonality_NEW_Spring,Seasonality_NEW_Summer,Seasonality_NEW_Winter
0,2022-01-01,S001,P0001,231,127,55,33.500,20,0,2022,1,1,5,26.800,3.810,1.128,nan,nan,nan,nan,nan,nan,nan,nan,nan,0,0,1,0,1,0,0,1,0,0,0,0,1
1,2022-01-02,S001,P0001,116,81,104,27.950,10,0,2022,1,2,6,25.155,-2.940,0.905,127.000,nan,nan,nan,nan,nan,nan,nan,nan,0,0,1,0,0,0,1,0,0,0,0,0,1
2,2022-01-03,S001,P0001,154,5,189,62.700,20,0,2022,1,3,0,50.160,4.480,1.077,81.000,127.000,nan,nan,nan,nan,nan,nan,nan,1,0,0,0,0,0,1,1,0,0,0,0,1
3,2022-01-04,S001,P0001,85,58,193,77.880,15,1,2022,1,4,1,66.198,1.890,1.025,5.000,81.000,127.000,nan,nan,nan,nan,nan,nan,0,0,1,0,0,1,0,0,0,0,0,0,1
4,2022-01-05,S001,P0001,238,147,37,28.460,20,1,2022,1,5,2,22.768,-0.940,0.968,58.000,5.000,81.000,nan,nan,nan,nan,nan,nan,0,0,1,0,0,1,0,0,0,1,0,0,1


In [ ]:
"""
for col in cat_features:

    train_categories = set(final_train_df[col].cat.categories)

    validation_unseen = (
        set(final_validation_df[col].dropna().unique())
        - train_categories
    )

    test_unseen = (
        set(final_test_df[col].dropna().unique())
        - train_categories
    )

    if validation_unseen:
        print(
            f"{col} - Validation'da train'de görülmeyen değerler: "
            f"{validation_unseen}"
        )

    if test_unseen:
        print(
            f"{col} - Test'te train'de görülmeyen değerler: "
            f"{test_unseen}"
        )
"""


In [133]:
# Outliers'ı Baskılama
def outlier_thresholds(dataframe, col_name, q1 = 0.25, q3 = 0.75):

    quartile1 = dataframe[col_name].quantile(q1)
    quartile3 = dataframe[col_name].quantile(q3)

    iqr = quartile3 - quartile1
    upper_limit = quartile3 + 1.5 * iqr
    lower_limit = quartile1 - 1.5 * iqr

    return lower_limit, upper_limit

def check_outliers( dataframe, col_name):
    lower_limit, upper_limit = outlier_thresholds(dataframe, col_name)
    if dataframe[(dataframe[col_name] > upper_limit) | (dataframe[col_name] < lower_limit)].any(axis=None):
        return True
    else:
        return False

def suppress_outliers(dataframe, col_name):
    low_limit, upper_limit = outlier_thresholds(dataframe, col_name)
    dataframe.loc[(dataframe[col_name] < low_limit), col_name] = low_limit
    dataframe.loc[(dataframe[col_name] > upper_limit), col_name] = upper_limit

In [134]:
num_cols

['Discount',
 'Inventory Level',
 'Units Ordered',
 'Price',
 'Month',
 'Day',
 'NET_PRICE',
 'units_sold_lag_1',
 'units_sold_lag_2',
 'units_sold_lag_3',
 'units_sold_lag_7',
 'units_sold_lag_14',
 'units_sold_lag_30',
 'units_sold_lag_365',
 'sales_roll_mean_7',
 'sales_roll_mean_30',
 'Year',
 'DayOfWeek',
 'Price_Diff',
 'Price_Ratio']

In [135]:
for col in num_cols:
    if check_outliers(final_train_df, col):
        print(f"{col} has outliers")

units_sold_lag_1 has outliers
units_sold_lag_2 has outliers
units_sold_lag_3 has outliers
units_sold_lag_7 has outliers
units_sold_lag_14 has outliers
units_sold_lag_30 has outliers
units_sold_lag_365 has outliers
sales_roll_mean_7 has outliers
sales_roll_mean_30 has outliers
Price_Ratio has outliers


In [136]:
final_train_df.head()
final_train_df.shape

(69900, 38)

In [137]:
final_test_df.head()
final_test_df.shape

(3200, 38)

In [ ]:
final_validation_df.head()
final_validation_df.shape

In [ ]:
# Preprocessingin son adımı olarak train ve test setten target'lar ayrılmalı

drop_cols = ['Date', 'Units Sold']

X_train = final_train_df.drop(columns=drop_cols)
y_train = final_train_df['Units Sold']

X_validation = final_validation_df.drop(columns=drop_cols)
y_validation = final_validation_df['Units Sold']

X_test = final_test_df.drop(columns=drop_cols)
y_test = final_test_df['Units Sold']

In [139]:
X_train.head()

,Store ID,Product ID,Inventory Level,Units Ordered,Price,Discount,Holiday/Promotion,Year,Month,Day,DayOfWeek,NET_PRICE,Price_Diff,Price_Ratio,units_sold_lag_1,units_sold_lag_2,units_sold_lag_3,units_sold_lag_7,units_sold_lag_14,units_sold_lag_30,units_sold_lag_365,sales_roll_mean_7,sales_roll_mean_30,Category_Electronics,Category_Furniture,Category_Groceries,Category_Toys,Region_North,Region_South,Region_West,Weather Condition_Rainy,Weather Condition_Snowy,Weather Condition_Sunny,Seasonality_NEW_Spring,Seasonality_NEW_Summer,Seasonality_NEW_Winter
0,S001,P0001,231,55,33.500,20,0,2022,1,1,5,26.800,3.810,1.128,nan,nan,nan,nan,nan,nan,nan,nan,nan,0,0,1,0,1,0,0,1,0,0,0,0,1
1,S001,P0001,116,104,27.950,10,0,2022,1,2,6,25.155,-2.940,0.905,127.000,nan,nan,nan,nan,nan,nan,nan,nan,0,0,1,0,0,0,1,0,0,0,0,0,1
2,S001,P0001,154,189,62.700,20,0,2022,1,3,0,50.160,4.480,1.077,81.000,127.000,nan,nan,nan,nan,nan,nan,nan,1,0,0,0,0,0,1,1,0,0,0,0,1
3,S001,P0001,85,193,77.880,15,1,2022,1,4,1,66.198,1.890,1.025,5.000,81.000,127.000,nan,nan,nan,nan,nan,nan,0,0,1,0,0,1,0,0,0,0,0,0,1
4,S001,P0001,238,37,28.460,20,1,2022,1,5,2,22.768,-0.940,0.968,58.000,5.000,81.000,nan,nan,nan,nan,nan,nan,0,0,1,0,0,1,0,0,0,1,0,0,1


In [ ]:

import pyarrow
# işlenen veriyi kaydet
final_train_df.to_parquet('../data/processed/train.parquet',engine='pyarrow',compression='snappy')

final_validation_df.to_parquet('../data/processed/validation.parquet',engine='pyarrow',compression='snappy')

final_test_df.to_parquet('../data/processed/test.parquet',engine='pyarrow',compression='snappy')


In [141]:
final_train_df.to_csv('../data/processed/train.csv', index=False)
final_test_df.to_csv('../data/processed/test.csv', index=False)

 !Önemli Not!

X_train/y_train/X_test/y_test diske kaydedilmiyor, sadece final_train_df/final_test_df kaydediliyor. Modelleme notebook'una geçildiğinde bu parquet dosyalarını okuyup drop_cols'u  (Date + Units Sold)'a tekrar uygulamak gerekecek 

DATE column'u özellikle tuttum. UI'da merge ile uğrşaılmadan DATE kırılımndan bi şeyler gösterilebilir.

